<div style="width: 100%; text-align: center; margin-bottom: 20px;">
  <img src="cabecalho_branco.jpg" alt="Cabeçalho" style="max-width: 100%; height: auto;">
</div>

# Cassarino

Autor: **Carlos Gabriel de Oliveira Campos**

Professor Dr. **Daniel Roberto Cassar**


Seja bem-vindo! O Cassarino é uma das maiores casas de jogos de probabilidade do reino de Lumi (não é uma casa de jogos de azar pois é cobrada uma entrada e não existem prêmios físicos, apenas pontos que não valem nada e são somados e exibidos em um telão central). Uma das máquinas mais famosas do Cassarino é o Cassar Níquel. O conceito 
do Cassarino e o Cassar Níquel foi uma criação de Luiza Davoli da Turma 25 (da Ilum de verdade!). O Cassar Níquel é uma máquina com 3  carretéis independentes. Ao ser ativada, cada carretel pode parar em um dos 5 símbolos: Space Invader (👾), Bolo (🍰), Bom Garoto (🐶), Vida Longa e Próspera (🖖) ou Maritaca (🦜).

---

## Contextualização

Neste trabalho, serão resolvidos problemas de probabilidade a partir do *Método de Monte Carlo* (MMC). Esse método se baseia na estimativa do valor da probabilidade de um evento a partir de amostragens aleatórias ou uma grande amostra de dados, que se fundamentam na Lei dos Grandes Números. Geralmente, a intenção ou motivação para a aplicação de MMC envolve a modelagem de um sistema do mundo real que não pode ser analisado por uma abordagem analítica ou tradicionalmente determinística [1], o que não é o caso do Cassarino, pois é possível calcular a probabilidade real dos eventos a serem descritos a seguir.

Dessa forma, serão geradas 10 milhões de simulações computacionais para obter a estimativa de probabilidade correspondente a cada cenário. Dado o tamanho da amostra $N$, um certo evento $A$ e o número de ocorrências de $A$, denotado por $n_A$, a probabilidade estimada para o evento ocorrer - $\hat{P}(A)$ - se dá conforme

$$
\hat{P}(A) = \frac{n_A}{N}.
$$

### Uma função para as simulações

Primeiro será criada a função `cassar_niquel` que simula uma roletada no cassar-níquel ao retornar uma lista com os resultados de cada roletada, cujo tipo é `tuple` contendo o que foi apresentado em cada um ods três carretéis na forma de `string`. Como argumentos, são obrigatórios uma sequência - como uma lista - dos eventos possíveis, assim como outra sequência com as probabilidades associadas a cada evento. Já como opcional, um inteiro que representa quantas roletadas serão simuladas, cujo valor padrão é apenas 1. Esse sistema é baseado na função `choices` do módulo embutido `random`, o qual faz escolhas pseudo-aleatórias para definir o resultado de cada um dos três carreteis de cada roletada. Assim, visando a reprodutibildiade dos resultados, será configurada uma semente aleatória de valor 42. 

OBS: Por se tratar de uma função relativamente simples para uso apenas nesse notebook, um filtro de argumentos foi dispensado. 

In [34]:
import random
from collections.abc import Sequence
from fractions import Fraction
from itertools import batched

In [35]:
def cassar_niquel[T](
    eventos: Sequence[T],
    probabilidades: Sequence[float | Fraction],
    n=1
    ) -> list[tuple[str]]:

    # Realiza todos os sorteios para cada carretel
    roletadas = random.choices(eventos, weights=probabilidades, k=3*n)

    # Divide e retorna a lista de sorteios de três em três pedaços (cada roletada)
    return [tuple(batch) for batch in batched(roletadas, 3)]

### Simulando uma única roletada

In [36]:
# Definindo semente aleatória
random.seed(42)

# Listas dos eventos e dos pesos correspondentes em ordem
lista_resultados = ["Space Invader", "Bolo", "Bom Garoto", "Vida Longa e Próspera", "Maritaca"]
lista_probabilidades = [0.35, 0.3, 0.2, 0.1, 0.05]

# Rodando a função
uma_unica_roletada = cassar_niquel(
    lista_resultados, 
    lista_probabilidades
)

uma_unica_roletada[0]

('Bolo', 'Space Invader', 'Space Invader')

Como podemos observar, essa roletada singular apresentou, *da esquerda para a direita*, os seguintes resultados:  
`('Bolo', 'Space Invader', 'Space Invader')`.

### Simulando **10 MILHÕES** de roletadas

A seguinte amostra gerada contendo 10 milhões de simulações será utilizada para todo o restante deste notebook.

In [37]:
N_SIMULACOES = 10_000_000

roletadas = cassar_niquel(
    lista_resultados, 
    lista_probabilidades, 
    N_SIMULACOES
    )

## Partindo para os problemas

Aliada à função `cassar_niquel`, será definida a função `print_pctg` para melhorar a visualização das probabilidades estimadas ao apresentar a probabildiade calculada, estimada e a precisão da estimativa em porcentagem com 4 algarismos "significativos" (de acordo com a formatação `f{valor:.{4}g}`) a fim de melhorar a visualização e comparação dos resultados obtidos.

In [38]:
def print_pctg(
    prob_calculada: float, prob_estimada: float):
    print(
        f"Probabilidade calculada: {100 * prob_calculada:.{4}g}%",
        f"Probabilidade estimada: {100 * prob_estimada:.{4}g}%",
        f"Precisão: {100 * prob_estimada / prob_calculada:.{4}g}%",
        sep="\n"
        )

### Visualizando a probabilidade de cada item em cada carretel

In [39]:
import pandas as pd

In [40]:
df = pd.DataFrame({
    "Resultado": lista_resultados,
    "Probabilidade": lista_probabilidades
})

print(df.to_string(index=False))

            Resultado  Probabilidade
        Space Invader           0.35
                 Bolo           0.30
           Bom Garoto           0.20
Vida Longa e Próspera           0.10
             Maritaca           0.05


### 1. Qual a probabilidade de sair pelo menos 1 🖖 entre os 3 carretéis?

In [41]:
contador_vida_longa_e_prospera: int = 0

for roletada in roletadas:
    if "Vida Longa e Próspera" in roletada:
        contador_vida_longa_e_prospera+=1

prob_estimada = contador_vida_longa_e_prospera / N_SIMULACOES
prob_calculada = (1 - (1 - 0.1)**3)

print_pctg(prob_calculada, prob_estimada)

Probabilidade calculada: 27.1%
Probabilidade estimada: 27.08%
Precisão: 99.93%


### 2. Qual a probabilidade de sair pelo menos 1 🐶, mas nenhum 🍰?

In [42]:
um_mais_garoto_nenhum_bolo: int = 0

for roletada in roletadas:
    if "Bom Garoto" in roletada and "Bolo" not in roletada:
        um_mais_garoto_nenhum_bolo += 1

prob_estimada = um_mais_garoto_nenhum_bolo / N_SIMULACOES
prob_calculada =  (1 - 0.3)**3 - (1 - 0.3 - 0.2)**3

print_pctg(prob_calculada, prob_estimada)

Probabilidade calculada: 21.8%
Probabilidade estimada: 21.78%
Precisão: 99.91%


### 3. Qual a probabilidade de sair exatamente 2 👾 entre os 3 carretéis?

In [43]:
dois_spaces_invaders: int = 0

for roletada in roletadas:
    if roletada.count("Space Invader") == 2:
        dois_spaces_invaders += 1

prob_estimada = dois_spaces_invaders / N_SIMULACOES
prob_calculada = (0.35)**2 * 0.65 * 3

print_pctg(prob_calculada, prob_estimada)

Probabilidade calculada: 23.89%
Probabilidade estimada: 23.88%
Precisão: 99.99%


### 4. Qual a probabilidade de sair exatamente a sequência 🦜‑🦜‑🦜?

In [44]:
tripla_maritaca: int = 0
EVENTO = ("Maritaca", "Maritaca", "Maritaca")

for roletada in roletadas:
    if roletada == EVENTO:
        tripla_maritaca += 1
        
prob_estimada = tripla_maritaca / N_SIMULACOES
prob_calculada = 0.05**3

print_pctg(prob_calculada, prob_estimada)

Probabilidade calculada: 0.0125%
Probabilidade estimada: 0.01221%
Precisão: 97.68%


### 5. Qual a probabilidade de sair pelo menos 2 símbolos iguais entre os 3 carretéis?

In [45]:
simbolo_repetido: int = 0

for roletada in roletadas:
    # como set(tuple) apaga os elementos repetidos, se o tamanho de set for menor que 3, então houve um símbolo repetido
    if len(set(roletada)) < 3:
        simbolo_repetido += 1

prob_estimada = simbolo_repetido / N_SIMULACOES

# Calculando probabilidade real
prob_calculada = 0
for p in lista_probabilidades:
    # Há dois casos: 3 ou exatamente 2 vezes o mesmo simbolo. Para 2 vezes, ha tres subscasos: AA_, _AA e A_A.
    prob_calculada += p**3 + 3*p**2*(1-p)

print_pctg(prob_calculada, prob_estimada)

Probabilidade calculada: 63.7%
Probabilidade estimada: 63.68%
Precisão: 99.96%


### 6. Jogando 20 vezes seguidas, qual a probabilidade de obter pelo menos um giro com 3 🐶?

In [46]:
roletadas20a20 = [list(batch) for batch in batched(roletadas, 20)]

EVENTO = ("Bom Garoto", "Bom Garoto", "Bom Garoto")
roletadas_com_triplo_garoto: int = 0

for grupo_20roletadas in roletadas20a20:
    for roletada in grupo_20roletadas:
        if roletada == EVENTO:
            roletadas_com_triplo_garoto += 1
            break

prob_estimada = roletadas_com_triplo_garoto / len(roletadas20a20)
prob_calculada = (1 - (1 - 0.2**3)**20)

print_pctg(prob_calculada, prob_estimada)

Probabilidade calculada: 14.84%
Probabilidade estimada: 14.79%
Precisão: 99.66%


### 7. Qual a probabilidade de sair 👾 no primeiro carretel e pelo menos uma 🦜 nos outros dois?

In [47]:
primeiro_space_maritaca_depois: int = 0

for roletada in roletadas:
    if roletada[0] == "Space Invader":
        if "Maritaca" in roletada[1:]:
            primeiro_space_maritaca_depois += 1

prob_estimada = primeiro_space_maritaca_depois / N_SIMULACOES
prob_calculada = 0.35 * (1 - (1 - 0.05)**2)

print_pctg(prob_calculada, prob_estimada)

Probabilidade calculada: 3.413%
Probabilidade estimada: 3.413%
Precisão: 100%


### 8. Qual a probabilidade de existir um 🍰 à direita de um 🖖?

In [48]:
bolo_direita_de_prosperidade = 0

for roletada in roletadas:
    # se o bolo não estiver nos ultimos dois, a condição não é possível
    if "Bolo" not in roletada[1:]:
        continue
    # para chegar aqui, ou o bolo está na 2ª ou na 3ª posição. então a mão deve estar ou na primeira ou na segunda 
    # posição para cumprir a condição
    if "Vida Longa e Próspera" in roletada[:2]:
        bolo_direita_de_prosperidade += 1

prob_estimada = bolo_direita_de_prosperidade / N_SIMULACOES
prob_calculada = 3 * (0.3 * 0.1) - (0.3 * 0.1 * 0.1) - (0.1 * 0.3 * 0.3 )

print_pctg(prob_calculada, prob_estimada)

Probabilidade calculada: 7.8%
Probabilidade estimada: 7.796%
Precisão: 99.95%


## Conclusão

Pôde-se observar o grande poder de precisão das simulações de Monte Carlo em relação ao valor real das probabilidades. O pior resultado obtido teve uma precisão de 97.68%, sendo que a maioria ultrapassou os 99%. Portanto, esse trabalho consolida, a partir da demonstração, a relevância do Método de Monte Carlo para o cálculo da estimativa de probabilidades. 

## Uso de IA neste trabalho

- Auxílio na otimização da função de sorteio;
- no cálculo das probabilidades;
- com tipagem no python;
- e compreensão de lista.

## Referências

[1] LI, Jiajie. Study of Monte Carlo Simulation: Principles, Methods, and Applications. Highlights in Science, Engineering and Technology, [s. l.], v. 140, p. 42, 2025. Disponível em https://doi.org/10.54097/sev38v22. Acesso em: 28 ago. 2026.

<div style="width: 100%; text-align: center; margin-bottom: 20px;">
  <img src="rodape_eleicao.png" alt="Rodapé" style="max-width: 100%; height: auto;">
</div>